# Micrograd: A Scalar Autodiff Engine

**Difficulty**: 🟡 Medium

### Problem Statement

`loss.backward()` is **reverse-mode automatic differentiation**. During the forward pass every operation records its inputs and how to push a gradient back through itself; `backward()` then walks the recorded graph in *reverse topological order*, applying the chain rule once per edge. Karpathy's *micrograd* shows the whole idea fits in ~100 lines of Python on scalars, and "implement autograd for scalars" is a classic research-engineer interview question because it tests the chain rule, graph traversal and Python operator overloading at once. PyTorch's autograd engine is the same algorithm with tensors as nodes and vector-Jacobian products as the local backward functions.

Build a `Value` class that behaves like a scalar `torch.tensor(x, dtype=torch.float64, requires_grad=True)`.

---

### Requirements

1. **`Value(data)`** — attributes `.data` (a Python `float`) and `.grad` (a Python `float`, starting at `0.0`).
2. **Arithmetic** producing new `Value`s: `a + b`, `a - b`, `a * b`, `a / b` and unary `-a`, where the other operand may be a `Value` **or a Python int/float on either side**: `v + 1`, `2 * v`, `1 - v`, `1 / v`, `3.0 + v` must all work (`__radd__`, `__rsub__`, `__rmul__`, `__rtruediv__`).
3. **Power by a constant**: `a ** k` for an int/float `k`, including negative and fractional exponents (e.g. `a ** -1.5` for `a > 0`). The exponent is never a `Value`.
4. **Functions**: `.exp()`, `.log()` (natural log), `.tanh()`, `.relu()`, `.sigmoid()`. `relu` has gradient **0 at exactly 0** (PyTorch's convention).
5. **`out.backward()`** sets `out.grad = 1.0` and propagates to every node `out` depends on, calling each node's local backward only after **all** of its consumers have contributed to its `.grad` (reverse topological order).
6. **Gradients accumulate**: a node used more than once (`x * x`, or `a = x + y; a * a`) receives the **sum** of the contributions from every use.
7. **No recursion limit**: `backward()` must work on a chain 5000 nodes deep at Python's default recursion limit (1000) — build the topological order with an explicit stack.
8. `backward()` is called once per freshly built graph (like PyTorch without `retain_graph=True`); you don't need to support calling it twice or zeroing gradients.

---

### Constraints

- ✅ Pure Python and the `math` module.
- ❌ No torch / numpy inside `Value` (torch is only used by the tests as the oracle).
- ❌ Don't make a recursive topological sort pass by raising `sys.setrecursionlimit` — use an iterative traversal.

---

<details>
  <summary>💡 Hint</summary>

  - Each operation computes `out.data`, remembers its inputs, and attaches a closure `out._backward` that knows the *local* derivatives and the chain rule: input.grad (something) local_derivative × out.grad. Should "(something)" be `=` or `+=`?
  - A topological order lists every node after all of its inputs; a post-order depth-first search produces one. To avoid recursion, push `(node, children_done)` pairs onto a list used as a stack.
  - You only need a few real primitives — `+`, `*`, `** k`, `exp`, `log`, `tanh`, `relu`, `sigmoid`. Express `-a`, `a - b` and `a / b` through them.
  - For `1 - v`, Python first tries `int.__sub__(1, v)`, which returns `NotImplemented`, and then calls `v.__rsub__(1)`. Mind the operand order.
  - Express derivatives through the *output* where that is cheaper: $\tanh' = 1 - \tanh^2$, $\sigma' = \sigma(1 - \sigma)$, $\exp' = \exp$.

</details>

---

In [ ]:
import math
import random
import sys

import torch   # the tests use torch.autograd as the oracle; Value itself must not use torch

In [ ]:
def torch_scalar(x):
    """A float64 leaf tensor that plays the role of Value(x) in the tests."""
    return torch.tensor(float(x), dtype=torch.float64, requires_grad=True)


print("default recursion limit:", sys.getrecursionlimit())

In [ ]:
class Value:
    """A scalar node in a computation graph, with reverse-mode autodiff."""

    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        # TODO: remember the inputs this node was computed from (_children), and set
        #       self._backward to a no-op closure (leaves have nothing to propagate to).
        #       Every operation below replaces _backward on the node it creates.

    # ---- primitives: each builds `out` and sets out._backward ------------------
    def __add__(self, other):          # self + other  (other: Value or int/float)
        # TODO
        raise NotImplementedError

    def __mul__(self, other):          # self * other
        # TODO
        raise NotImplementedError

    def __pow__(self, k):              # self ** k, k a constant int/float
        # TODO
        raise NotImplementedError

    def exp(self):
        # TODO
        raise NotImplementedError

    def log(self):
        # TODO
        raise NotImplementedError

    def tanh(self):
        # TODO
        raise NotImplementedError

    def relu(self):                    # gradient 0 at exactly 0
        # TODO
        raise NotImplementedError

    def sigmoid(self):
        # TODO
        raise NotImplementedError

    # ---- derived ops: express through the primitives above --------------------
    def __neg__(self):                 # -self
        # TODO
        raise NotImplementedError

    def __sub__(self, other):          # self - other
        # TODO
        raise NotImplementedError

    def __truediv__(self, other):      # self / other
        # TODO
        raise NotImplementedError

    def __radd__(self, other):         # other + self, other an int/float
        # TODO
        raise NotImplementedError

    def __rsub__(self, other):         # other - self
        # TODO
        raise NotImplementedError

    def __rmul__(self, other):         # other * self
        # TODO
        raise NotImplementedError

    def __rtruediv__(self, other):     # other / self
        # TODO
        raise NotImplementedError

    # ---- reverse-mode sweep ---------------------------------------------------
    def backward(self):
        """Seed self.grad = 1.0 and run every node's _backward in reverse topological order.

        Build the topological order WITHOUT recursion (explicit stack).
        """
        # TODO
        raise NotImplementedError

    def __repr__(self):
        return f"Value(data={self.data:.6g}, grad={self.grad:.6g})"

In [ ]:
# Test 1 — a small expression differentiated by hand
a, b = Value(2.0), Value(-3.0)
c = a * b + b ** 2              # -6 + 9 = 3
c.backward()
assert c.data == 3.0, c
assert a.grad == -3.0, f"dc/da = b = -3, got {a.grad}"
assert b.grad == -4.0, f"dc/db = a + 2b = -4, got {b.grad}"
print("hand-derived gradients OK:", a, b)

In [ ]:
# Test 2 — Python numbers on either side of an operator
for expr, f in [("2 * v", lambda v: 2 * v), ("3 + v", lambda v: 3 + v), ("1 - v", lambda v: 1 - v),
                ("1 / v", lambda v: 1 / v), ("v / 4", lambda v: v / 4), ("v - 2", lambda v: v - 2),
                ("v ** -1.5", lambda v: v ** -1.5), ("-v", lambda v: -v)]:
    v, t = Value(4.0), torch_scalar(4.0)
    out_v, out_t = f(v), f(t)
    out_v.backward()
    out_t.backward()
    assert abs(out_v.data - out_t.item()) < 1e-12, f"{expr}: value {out_v.data} vs {out_t.item()}"
    assert abs(v.grad - t.grad.item()) < 1e-12, f"{expr}: grad {v.grad} vs {t.grad.item()}"
print("reflected ops OK")

In [ ]:
# Test 3 — a node used several times must receive the SUM of its gradient contributions
x = Value(3.0)
y = x * x
y.backward()
assert x.grad == 6.0, f"d(x*x)/dx = 2x = 6, got {x.grad}"

x, y = Value(1.5), Value(-0.5)
a = x + y
out = a * a                     # = (x + y)^2
out.backward()
assert x.grad == 2.0 and y.grad == 2.0, f"expected 2(x+y) = 2 for both, got {x.grad}, {y.grad}"
print("shared nodes accumulate OK")

In [ ]:
# Test 4 — every op in one expression with shared subexpressions, vs torch.autograd
def expression(x, y, z):
    a = x * y + z.exp()               # a is used three times below
    b = (a / (1 + y ** 2)).tanh()
    c = (z ** 2 + 0.5).log() - a.sigmoid() * 2
    d = (b - c).relu() + (1 - x) / 3
    return d * a + b ** -2 + (-c) * a


inputs = (0.6, -1.2, 0.3)
vals = [Value(v) for v in inputs]
tens = [torch_scalar(v) for v in inputs]
out_v, out_t = expression(*vals), expression(*tens)
out_v.backward()
out_t.backward()
print(f"output: Value {out_v.data:.10f}   torch {out_t.item():.10f}")
assert abs(out_v.data - out_t.item()) < 1e-9
for name, v, t in zip("xyz", vals, tens):
    print(f"d/d{name}: Value {v.grad:+.10f}   torch {t.grad.item():+.10f}")
    assert abs(v.grad - t.grad.item()) < 1e-9, f"wrong gradient for {name}"

In [ ]:
# Test 5 — a 5000-deep chain at Python's default recursion limit
old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    x = Value(0.5)
    y = x
    for _ in range(5000):
        y = y + x                     # y = 5001 * x
    y.backward()
finally:
    sys.setrecursionlimit(old_limit)
assert x.grad == 5001.0, f"expected 5001, got {x.grad}"
print("deep chain OK:", x)

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("micrograd", Value=Value)